# Cardiomyocyte size pipeline -- demo notebook

End-to-end walkthrough of the same pipeline that lives in `src/`:

1. Visualise the segmentation step-by-step on a single image (sanity check).
2. Run the batch over a folder of `.lif` files and write a per-image CSV.
3. Plot the distributions, the count-vs-area correlation, and (optionally) a group comparison.

Segmentation uses [Cellpose](https://github.com/MouseLand/cellpose) on the WGA (cell borders) and DAPI (nuclei) channels.

## 0. Install dependencies

Skip this cell if you've already installed `requirements.txt` in your environment.

In [ ]:
# !pip install -r ../requirements.txt

In [ ]:
# Make the local `src` package importable when running from the notebooks/ folder.
import os, sys
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from src.segmentation import SegmentationConfig
from src.visualization import visualize_pipeline
from src.batch import run_batch
from src.plotting import (
    load_results, add_sample_name,
    plot_distributions, plot_count_vs_area,
    plot_group_comparison, load_group_mapping,
)

## 1. Step-by-step visualisation on one image

Useful before launching a long batch -- confirm the channel indices are right and the Cellpose parameters give sensible masks on a representative image.

In [ ]:
# Point this at one of your own .lif files.
LIF_FILE_PATH = '/path/to/your_file.lif'
IMAGE_INDEX = 0

cfg = SegmentationConfig(
    dapi_channel=0,
    wga_channel=2,
    z_strategy='max',
    diameter=200,
    flow_threshold=0.8,
    cellprob_threshold=-2.0,
    min_area_um2=160,
)

fig = visualize_pipeline(LIF_FILE_PATH, image_index=IMAGE_INDEX, cfg=cfg)
fig.show()

## 2. Batch processing

Walks a folder of `.lif` files and writes one row per image. Cellpose is loaded once and reused across all images.

In [ ]:
INPUT_DIR  = '/path/to/lif_files'
OUTPUT_CSV = '../results/per_image_results.csv'

df_results = run_batch(input_dir=INPUT_DIR, output_csv=OUTPUT_CSV, cfg=cfg)
df_results.head()

## 3. Distributions and quality control

Drops images with very few detected cells (typically out-of-focus or empty fields), then plots histograms of cell count and mean area.

In [ ]:
df_clean = load_results(OUTPUT_CSV, min_cells_per_image=10)
df_clean = add_sample_name(df_clean)

plot_distributions(df_clean);

## 4. Cell count vs. mean area

A negative correlation here is expected and reassuring: when cells are bigger, fewer fit in a fixed-size field of view.

In [ ]:
plot_count_vs_area(df_clean);

## 5. Group comparison (optional)

Provide a CSV mapping each sample to a group label (see `config/sample_groups.example.csv` for the format) and a list specifying the desired x-axis order.

In [ ]:
GROUPS_CSV  = '../config/sample_groups.example.csv'
GROUP_ORDER = ['Group_A', 'Group_B', 'Group_C']
GROUP_COLORS = {
    'Group_A': '#488f31',
    'Group_B': '#329db3',
    'Group_C': '#de425b',
}

sample_to_group = load_group_mapping(GROUPS_CSV)

# One dot per sample (biological replicate).
plot_group_comparison(
    df_clean=df_clean,
    sample_to_group=sample_to_group,
    group_order=GROUP_ORDER,
    group_colors=GROUP_COLORS,
    aggregate_by_sample=True,
);

In [ ]:
# One dot per image (useful for QC, lots more datapoints).
plot_group_comparison(
    df_clean=df_clean,
    sample_to_group=sample_to_group,
    group_order=GROUP_ORDER,
    group_colors=GROUP_COLORS,
    aggregate_by_sample=False,
);